# PLL lock time

Results of `pll_lock_time.lock_time_sweep` (`sw/tests/pll_lock_time.py`): the reference hops by `step_rel` with FSK, the MXR058A captures the divided VCO on the pad triggered on the FG Sync output, and the VCO frequency against time is averaged over the captures. Per point and step direction (up / down): the settled frequencies, the measured step against `FB_DIV` x the reference step, and the time after the step from which the frequency stays inside each band (10 % / 1 % of the step, 0.1 % / 0.01 % of the frequency).

In [29]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
import plotly.io as pio
from plotly.subplots import make_subplots

# VS Code: the default renderer needs the jupyter-renderers extension ("Error loading renderer" when it fails
# to load); plain HTML with plotly.js from the CDN shows without it. HTML and PNG are saved next to the CSV anyway
pio.renderers.default = 'notebook_connected'

PLL_DIR = Path.cwd().resolve().parents[2] / 'results' / 'pll'  # notebook lives in sw/tools/notebooks/
# The run to plot; None = the newest pll_lock_time_*.csv
RUN_CSV = None
# Bands of SETTLE_BANDS (pll_lock_time.py): column suffix -> (kind, value), as in the script
BANDS = {'10pct': ('step', 0.1), '1pct': ('step', 0.01), '1000ppm': ('rel', 1e-3), '100ppm': ('rel', 1e-4)}
# Display names of the bands ('rel' bands in % of the frequency; the CSV column names keep ppm)
BAND_TXT = {'10pct': '10pct', '1pct': '1pct', '1000ppm': '0.1 %', '100ppm': '0.01 %'}
ZOOM_US = 3  # [us] after the step shown in the zoomed plot (the PLL settles in ~0.2-0.5 us)


def newest_run():
    return sorted((p for p in PLL_DIR.glob('pll_lock_time_*.csv') if not p.stem.endswith('_traces')),
                  key=lambda p: p.stat().st_mtime)[-1]


RUN_CSV = Path(RUN_CSV) if RUN_CSV is not None else newest_run()
rows = pd.read_csv(RUN_CSV)
traces_path = RUN_CSV.with_name(RUN_CSV.stem + '_traces.csv')
traces = pd.read_csv(traces_path) if traces_path.exists() and traces_path.stat().st_size else pd.DataFrame()
RUN_TXT = RUN_CSV.stem
measured = rows[rows['slope'].notna()].reset_index(drop=True)  # not_locked / error points have no slope

table = pd.DataFrame({
    'point': rows['label'],
    'step': rows['direction'],
    'status': rows['status'],
    'VCO [GHz]': (rows['f_vco'] / 1e9).round(4),
    'codes c/min/max': rows[['vco_tune_coarse', 'vco_current_min', 'vco_current_max']].astype('Int64').astype(str).fillna('-')
                       .agg('/'.join, axis=1),
    'Icp/C1/C2/R1': rows[['set_current', 'set_c1', 'set_c2', 'set_r1']].astype('Int64').astype(str).fillna('-').agg('/'.join, axis=1),
    'locked / hopping / end': rows[['locked', 'locked_fsk', 'locked_end']].astype('Int64').astype(str).fillna('-')
                              .agg('/'.join, axis=1),
    'captures': rows['n_valid'].astype('Int64').astype(str) + '/' + rows['n_captures'].astype('Int64').astype(str),
    'step [MHz]': (rows['step_meas'] / 1e6).round(3),
    'expected [MHz]': (rows['step_expected'] / 1e6).round(3),
    'noise [%]': (rows['noise_ppm'] / 1e4).round(4),
    'rise 10-90 [ns]': (rows.get('t_rise_10_90', np.nan) * 1e9).round(1),  # not in files of older runs
    'overshoot [%]': rows.get('overshoot_pct', np.nan).round(1) if 'overshoot_pct' in rows else np.nan,
    'peak at [ns]': (rows.get('t_peak', np.nan) * 1e9).round(1),
    **{f't {BAND_TXT[b]} [us]': (rows[f't_settle_{b}'] * 1e6).round(2) for b in BANDS},
    't 10% per capture [us]': ((rows['t_settle_10pct_capt_median'] * 1e6).round(2).astype(str) + ' ± '
                               + (rows['t_settle_10pct_capt_std'] * 1e6).round(2).astype(str)),
    'P [mW]': (rows['pll_p'] * 1e3).round(3),
})
print(f'{RUN_TXT}: {len(rows)} rows, {len(measured)} measured slopes; sample rate '
      f"{rows['sample_rate'].dropna().unique() / 1e6} MSa/s, pad {rows['f_pad'].dropna().unique() / 1e6} MHz")
print('t <band> empty: not settled inside the window (or not measured); a band below ~3x the noise only measures '
      'the noise')
display(table)


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to RUN_CSV (<stem>_<name>), then show the figure."""
    base = RUN_CSV.with_name(f'{RUN_CSV.stem}_{name}')
    fig.write_html(base.with_suffix('.html'))
    try:
        fig.write_image(base.with_suffix('.png'), width=1100, height=fig.layout.height or 600, scale=2)
        print(f'saved {base}.html and .png')
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f'saved {base}.html; PNG export failed: {e}')
    fig.show()


def curve_of(r):
    """The averaged curve of a row (point label and slope), sorted in time."""
    if traces.empty:
        return traces
    return traces[(traces['label'] == r['label']) & (traces['slope'] == r['slope'])].sort_values('t')

pll_lock_time_S5_20261109_230454: 2 rows, 2 measured slopes; sample rate [16000.] MSa/s, pad [100.] MHz
t <band> empty: not settled inside the window (or not measured); a band below ~3x the noise only measures the noise


,point,step,status,VCO [GHz],codes c/min/max,Icp/C1/C2/R1,locked / hopping / end,captures,step [MHz],expected [MHz],noise [%],rise 10-90 [ns],overshoot [%],peak at [ns],t 10pct [us],t 1pct [us],t 0.1 % [us],t 0.01 % [us],t 10% per capture [us],P [mW]
0,"1/1 4.800 GHz nominal, step 5 % (step 5.0 %)",up,ok,4.8,0/8/8,7/6/0/7,1/1/1,1000/1000,239.882,240.0,0.0072,30.2,39.2,85.0,0.22,0.55,0.29,NaN,3.13 ± 0.54,NaN
1,"1/1 4.800 GHz nominal, step 5 % (step 5.0 %)",down,ok,4.8,0/8/8,7/6/0/7,1/1/1,1000/1000,-239.929,-240.0,0.0042,19.0,44.6,85.0,0.14,1.98,1.14,NaN,1.58 ± 0.73,NaN


## Settling after the step

VCO frequency minus the settled value (`f_after`) against the time after the Sync edge, averaged over the captures (shaded: ± the spread of the single captures). Left: the whole window; right: the first `ZOOM_US` µs. Dotted: ±10 % and ±1 % of the step; vertical lines: the settling times to those bands.

In [30]:
COLORS = {'up': '#d1495b', 'down': '#2a6fdb'}

for label, point in measured.groupby('label', sort=False):
    fig = make_subplots(rows=1, cols=2, subplot_titles=('whole window', f'first {ZOOM_US} µs'))
    for _, r in point.iterrows():
        c = curve_of(r)
        if c.empty:
            continue
        color, t_us = COLORS.get(r['direction'], 'gray'), c['t'] * 1e6
        err = (c['f_vco_mean'] - r['f_after']) / 1e6
        for col in (1, 2):
            fig.add_trace(go.Scatter(x=pd.concat([t_us, t_us[::-1]]),
                                     y=pd.concat([err + c['f_vco_std'] / 1e6, (err - c['f_vco_std'] / 1e6)[::-1]]),
                                     fill='toself', fillcolor=color, opacity=0.15, line=dict(width=0),
                                     hoverinfo='skip', showlegend=False), row=1, col=col)
            fig.add_trace(go.Scatter(x=t_us, y=err, mode='lines', line=dict(color=color, width=1.5),
                                     name=f"{r['direction']} ({r['step_meas'] / 1e6:+.1f} MHz)", showlegend=col == 1,
                                     hovertemplate='%{x:.2f} us<br>%{y:.3f} MHz<extra></extra>'), row=1, col=col)
            for b, dash in (('10pct', 'dot'), ('1pct', 'dash')):
                t = r[f't_settle_{b}'] * 1e6
                if not np.isnan(t) and (col == 1 or t <= ZOOM_US):
                    fig.add_vline(x=t, line=dict(color=color, dash=dash, width=1), row=1, col=col)
    step = point['step_meas'].abs().max() / 1e6
    for frac in (0.1, 0.01):
        for sign in (1, -1):
            fig.add_hline(y=sign * frac * step, line=dict(color='gray', dash='dot', width=1))
    fig.update_xaxes(title='time after the step [µs]')
    fig.update_xaxes(range=[-0.1 * ZOOM_US, ZOOM_US], row=1, col=2)
    fig.update_yaxes(title='f_vco − settled [MHz]', row=1, col=1)
    fig.update_layout(title=f'Settling, {label} ({RUN_TXT})', height=500, template='plotly_white')
    save(fig, 'settling_' + label.split()[0].replace('/', 'of'))

saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_settling_1of1.html and .png


## Frequency error on a log scale

|f_vco − settled| / settled [%] after the step: where the curve meets the 0.1 % / 0.01 % lines is the settling time to those bands; the floor it ends on is the measurement noise (`noise_ppm`, shown in %, dash-dot).

In [31]:
for label, point in measured.groupby('label', sort=False):
    fig = go.Figure()
    for _, r in point.iterrows():
        c = curve_of(r)
        if c.empty:
            continue
        c = c[c['t'] >= 0]
        color = COLORS.get(r['direction'], 'gray')
        fig.add_trace(go.Scatter(x=c['t'] * 1e6, y=(c['f_vco_mean'] - r['f_after']).abs() / r['f_after'] * 100,
                                 mode='lines', line=dict(color=color, width=1.5), name=r['direction'],
                                 hovertemplate='%{x:.2f} us<br>%{y:.4f} %<extra></extra>'))
        fig.add_hline(y=r['noise_ppm'] / 1e4, line=dict(color=color, dash='dashdot', width=1))
    for b, (kind, value) in BANDS.items():
        if kind == 'rel':
            fig.add_hline(y=value * 100, line=dict(color='gray', dash='dot', width=1), annotation_text=BAND_TXT[b],
                          annotation_position='top right')
    fig.update_xaxes(title='time after the step [µs]')
    fig.update_yaxes(title='|frequency error| [%]', type='log', range=[-4, 1])  # 0.0001 % - 10 %, fixed: hline annotations broke the autorange
    fig.update_layout(title=f'Frequency error, {label} ({RUN_TXT})', height=450, template='plotly_white')
    save(fig, 'error_' + label.split()[0].replace('/', 'of'))

saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_error_1of1.html and .png


## Lock time per capture (10 %, noise-limited)

Settling time to 10 % of the step of every single capture (`t_settle_10pct_capt`). At 10 ns bins the noise of one capture (about 14 MHz per bin) is as large as the 10 % band of a 3 % step (14.4 MHz), so most captures register a late excursion: NaN or a value in the µs. Only the 5 % step has a band wider than the noise, and its spread is where the noise crosses the band, not capture-to-capture jitter of the loop. Use the 50 % crossing below for a per-capture quantity.

In [32]:
fig = go.Figure()
for _, r in measured.iterrows():
    vals = pd.to_numeric(pd.Series(str(r['t_settle_10pct_capt']).split()), errors='coerce') * 1e6
    fig.add_trace(go.Box(y=vals, name=f"{r['label']} {r['direction']}", boxpoints='all', jitter=0.4, pointpos=0,
                         marker=dict(color=COLORS.get(r['direction'], 'gray'), size=6),
                         line=dict(color=COLORS.get(r['direction'], 'gray'))))
    n_nan = int(vals.isna().sum())
    if n_nan:
        print(f"{r['label']} {r['direction']}: {n_nan} of {len(vals)} captures not settled inside the window")
fig.update_yaxes(title='lock time to 10 % of the step [µs]', rangemode='tozero')
fig.update_layout(title=f'Lock time per capture ({RUN_TXT})', height=450, template='plotly_white', showlegend=False)
save(fig, 'captures')

1/1 4.800 GHz nominal, step 5 % (step 5.0 %) up: 809 of 1000 captures not settled inside the window
1/1 4.800 GHz nominal, step 5 % (step 5.0 %) down: 135 of 1000 captures not settled inside the window
saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_captures.html and .png


## Pad period lengths

Histograms of the period lengths of the pad clock over the last long free-running pad record without FSK (`pll_lock_time.long_pad_record`, `PAD_RECORD_S` long, about 6000 periods per edge type at 200 µs; saved in `pll_lock_time_pad_record.npz`, the newest run): rising edge to rising edge (red) and falling edge to falling edge (blue), each with a Gaussian of its own mean and standard deviation. The crossings are taken at the mid level (between the 5th and 95th percentile of the record) and interpolated between samples, as in the script. Several separate peaks instead of one Gaussian: the pad edges are displaced in a repeating pattern (pickup of the reference, not jitter); the width of a single peak is the random jitter plus the scope noise.

In [33]:
REC = PLL_DIR / 'pll_lock_time_pad_record.npz'  # overwritten by every run: the newest pad record
rec = np.load(REC)
t_rec, v_rec = rec['t'], rec['v']
lo, hi = np.percentile(v_rec, [5, 95])
mid = (lo + hi) / 2


def edge_times(rising):
    """Interpolated times [s] of the rising / falling crossings of the mid level; crossings closer than half the median
    period to the previous one (noise on an edge) dropped."""
    a, b = (v_rec[:-1] < mid, v_rec[1:] >= mid) if rising else (v_rec[:-1] >= mid, v_rec[1:] < mid)
    i = np.flatnonzero(a & b)
    tc = t_rec[i] + (mid - v_rec[i]) / (v_rec[i + 1] - v_rec[i]) * (t_rec[i + 1] - t_rec[i])
    gap = 0.5 * np.median(np.diff(tc))
    keep = [0]
    for k in range(1, len(tc)):
        if tc[k] - tc[keep[-1]] >= gap:
            keep.append(k)
    return tc[keep]


if 'rise' in rec.files and 'fall' in rec.files:  # the long record of long_pad_record (PAD_RECORD_S)
    rise, fall = rec['rise'], rec['fall']
else:  # an older file: only the short record
    rise, fall = edge_times(True), edge_times(False)
series = {'rising to rising': (np.diff(rise), '#d1495b'),
          'falling to falling': (np.diff(fall), '#2a6fdb')}
mu_all = np.mean([x.mean() for x, _ in series.values()])  # common zero: the pad period
fig = go.Figure()
bin_w = None
for name, (per, color) in series.items():
    mu, sigma = per.mean(), per.std()
    print(f'{name}: {len(per)} periods, mean {mu * 1e9:.4f} ns ({1 / mu / 1e6:.4f} MHz), std {sigma * 1e12:.1f} ps, '
          f'min/max {per.min() * 1e9:.3f} / {per.max() * 1e9:.3f} ns')
    x_ps = (per - mu_all) * 1e12
    n_bins = int(np.clip(np.sqrt(len(per)) * 2, 20, 200))
    fig.add_trace(go.Histogram(x=x_ps, nbinsx=n_bins, name=name, opacity=0.5, marker=dict(color=color)))
    bin_w = (x_ps.max() - x_ps.min()) / n_bins
    grid = np.linspace(x_ps.min() - 3 * sigma * 1e12, x_ps.max() + 3 * sigma * 1e12, 600)
    gauss = len(per) * bin_w / (sigma * 1e12 * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((grid - (mu - mu_all) * 1e12) / (sigma * 1e12)) ** 2)
    fig.add_trace(go.Scatter(x=grid, y=gauss, mode='lines', line=dict(color=color, width=2),
                             name=f'Gaussian {name}: sigma {sigma * 1e12:.1f} ps'))
fig.update_xaxes(title=f'period - {mu_all * 1e9:.4f} ns [ps]')
fig.update_yaxes(title='periods')
fig.update_layout(title=f'Pad period lengths ({REC.name})', height=450, template='plotly_white', barmode='overlay')
save(fig, 'period_hist')


rising to rising: 19999 periods, mean 10.0000 ns (100.0002 MHz), std 548.3 ps, min/max 8.996 / 10.628 ns
falling to falling: 19999 periods, mean 10.0000 ns (100.0002 MHz), std 604.2 ps, min/max 8.887 / 10.844 ns
saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_period_hist.html and .png


## Overview over the data set

Per point (the factor changed from the reference point, in the label's brackets) and step direction: the rise time (10 to 90 %), the overshoot and the settling time to ±5 % of the step. For the step-size, loop-filter (`set_current`, `set_r1`, `set_c1`) and VCO-frequency points of `pll_lock_time.DATA_POINTS`. Missing bars: the point did not lock / no valid result, or the column is not in the file of an older run.

In [34]:
needed = ['t_rise_10_90', 'overshoot_pct', 't_settle_5pct']
if all(c in measured.columns for c in needed) and len(measured):
    ov = measured.copy()
    ov['point'] = ov['label'].str.extract(r'\((.*)\)')[0].fillna(ov['label'])
    fig = make_subplots(rows=1, cols=3, subplot_titles=('rise time 10-90 % [ns]', 'overshoot [% of the step]',
                                                        'settling to ±5 % of the step [ns]'))
    for col, (name, scale) in enumerate((('t_rise_10_90', 1e9), ('overshoot_pct', 1), ('t_settle_5pct', 1e9)), 1):
        for direction, color in COLORS.items():
            sub = ov[ov['direction'] == direction]
            fig.add_trace(go.Bar(x=sub['point'], y=sub[name] * scale, name=direction, marker=dict(color=color),
                                 showlegend=(col == 1)), row=1, col=col)
    fig.update_layout(height=450, template='plotly_white', barmode='group', title=f'Data set overview ({RUN_TXT})')
    fig.update_xaxes(tickangle=-45)
    save(fig, 'overview')
else:
    print('no overview: the run has no rise / overshoot / 5 % columns (run of before 2026-11-09 17:00?)')


saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_overview.html and .png


## Dead time and loop dynamics (fit)

Every averaged curve is fitted with the response of a type-II PLL to a frequency step, delayed by a dead time: with the natural frequency `fn`, the damping `zeta` (shown as the quality factor `Q = 1 / (2 zeta)`) and the dead time `t0`, the normalised response (0 before the step, 1 settled) is `1 - exp(-s t) (cos(wd t) - s/wd sin(wd t))` for `t > t0` (s = zeta wn, wd = wn sqrt(1 - zeta^2); the loop filter zero gives the `2 zeta wn s` term in the numerator, so the frequency starts to move linearly). The model is averaged over the width of a bin (`bin_s`) like the measurement. `t0` is the time after the Sync edge before the VCO frequency starts to move: the reference hop in the FG, the cables, the phase detector (up to one reference period until it sees the new edge) and the divider, not only the loop. A `t0` that does not depend on the step size is the loop's; a settling time that grows with the step is slewing, outside the linear model (the residual `rms` grows then). Plots against the step size; the table has the numbers.

In [35]:
FIT_WINDOW_NS = (-30, 600)  # [ns] of the curve fitted


def step_response(p, t):
    """Normalised frequency step response of a type-II PLL with dead time: p = (t0 [ns], wn [rad/ns], zeta), t [ns]."""
    t0, wn, z = p
    tt = np.clip(t - t0, 0, None)
    s = z * wn
    if z < 1:
        wd = wn * np.sqrt(1 - z * z)
        y = 1 - np.exp(-s * tt) * (np.cos(wd * tt) - (s / wd) * np.sin(wd * tt))
    else:
        wd = wn * np.sqrt(z * z - 1)
        y = 1 - np.exp(-s * tt) * (np.cosh(wd * tt) - (s / wd) * np.sinh(wd * tt))
    return np.where(t < t0, 0.0, y)


def fit_response(t, x, bin_ns, seed=0):
    """(t0 [ns], fn [MHz], zeta, rms of the residual in steps; the table and the plot show Q = 1 / (2 zeta)) of the best fit of `step_response`, averaged over the bin
    width, to the normalised curve x(t): a random search followed by a shrinking random refinement (numpy only)."""
    ok = (t > FIT_WINDOW_NS[0]) & (t < FIT_WINDOW_NS[1]) & ~np.isnan(x)
    t, x = t[ok], x[ok]
    if len(t) < 10:
        return np.nan, np.nan, np.nan, np.nan
    sub = np.linspace(-bin_ns / 2, bin_ns / 2, 9)

    def cost(p):
        return np.sum((np.mean([step_response(p, t + d) for d in sub], axis=0) - x) ** 2)

    rng = np.random.default_rng(seed)
    best = None
    for _ in range(4000):
        p = np.array([rng.uniform(-5, 40), rng.uniform(0.02, 0.25), rng.uniform(0.1, 1.2)])
        c = cost(p)
        if best is None or c < best[0]:
            best = (c, p)
    step = np.array([2.0, 0.02, 0.1])
    for it in range(400):
        c, p = best
        q = p + rng.normal(0, 1, 3) * step
        q[1], q[2] = max(q[1], 0.005), max(q[2], 0.05)
        cq = cost(q)
        if cq < c:
            best = (cq, q)
        if it % 100 == 99:
            step *= 0.5
    c, (t0, wn, z) = best
    return t0, wn / (2 * np.pi) * 1e3, z, np.sqrt(c / len(t))


# --- the fits, the table and the plots
fits = []
for _, r in measured.iterrows():
    c = curve_of(r)
    if c.empty or not np.isfinite(r['f_after'] - r['f_before']):
        continue
    t_ns = c['t'].to_numpy() * 1e9
    x = ((c['f_vco_mean'] - r['f_before']) / (r['f_after'] - r['f_before'])).to_numpy()
    t0, fn, zeta, rms = fit_response(t_ns, x, r['bin_s'] * 1e9)
    fits.append(dict(label=r['label'], direction=r['direction'], step_rel=r['step_rel'], t0_ns=t0, fn_MHz=fn, Q=1 / (2 * zeta), rms=rms))
fits = pd.DataFrame(fits)
display(fits.round({'step_rel': 4, 't0_ns': 1, 'fn_MHz': 2, 'Q': 2, 'rms': 3}))

if len(fits):
    fig = make_subplots(rows=1, cols=3, subplot_titles=('dead time t0 [ns]', 'natural frequency fn [MHz]', 'quality factor Q = 1 / (2 zeta)'))
    for col, name in enumerate(('t0_ns', 'fn_MHz', 'Q'), 1):
        for direction, color in COLORS.items():
            sub = fits[fits['direction'] == direction].sort_values('step_rel')
            fig.add_trace(go.Scatter(x=sub['step_rel'] * 100, y=sub[name], mode='lines+markers', name=direction,
                                     line=dict(color=color), showlegend=(col == 1)), row=1, col=col)
        fig.update_xaxes(title='reference step [%]', row=1, col=col)
    fig.update_layout(height=420, template='plotly_white', title=f'Dead time and loop dynamics ({RUN_TXT})')
    save(fig, 'dead_time')


,label,direction,step_rel,t0_ns,fn_MHz,Q,rms
0,"1/1 4.800 GHz nominal, step 5 % (step 5.0 %)",up,0.05,45.2,10.32,1.60,0.050
1,"1/1 4.800 GHz nominal, step 5 % (step 5.0 %)",down,0.05,38.6,10.95,1.85,0.026


saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_dead_time.html and .png


## Response time per capture (50 % crossing)

For every single capture: the time after the Sync edge at which its frequency curve first reaches 50 % of its own step (`t50_capt`, three-bin moving average, so the noise in the tail does not matter, unlike for a settling time). The spread between captures is the spread of the dead time (the phase detector sees the new reference edge anywhere within one reference period, since the FSK rate is not locked to it) plus the noise; the median is the response time to half the step. Missing points: the capture did not reach 50 % inside the window, or the column is not in the file of an older run.

In [36]:
if 't50_capt' in measured.columns:
    fig = go.Figure()
    for _, r in measured.iterrows():
        vals = pd.to_numeric(pd.Series(str(r['t50_capt']).split()), errors='coerce') * 1e9
        tag = r['label'].split('(')[-1].rstrip(')') if '(' in r['label'] else r['label']
        fig.add_trace(go.Box(y=vals, name=f"{tag} {r['direction']}", boxpoints='all', jitter=0.4, pointpos=0,
                             marker=dict(color=COLORS.get(r['direction'], 'gray'), size=5),
                             line=dict(color=COLORS.get(r['direction'], 'gray'))))
        print(f"{tag} {r['direction']}: median {vals.median():.1f} ns, std {vals.std():.1f} ns, "
              f"{int(vals.isna().sum())} of {len(vals)} captures without a 50 % crossing")
    fig.update_yaxes(title='time to 50 % of the step [ns]', rangemode='tozero')
    fig.update_layout(title=f'Response time per capture ({RUN_TXT})', height=450, template='plotly_white',
                      showlegend=False)
    save(fig, 't50_captures')
else:
    print('no t50_capt column: the run is from before the per-capture 50 % crossing was added')


step 5.0 % up: median 47.2 ns, std 13.1 ns, 0 of 1000 captures without a 50 % crossing
step 5.0 % down: median 45.4 ns, std 13.8 ns, 0 of 1000 captures without a 50 % crossing
saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_t50_captures.html and .png


## Effect of the pickup correction
The raw captures of the run (`<csv stem>_waveforms/`) processed twice with the analysis functions of `pll_lock_time.py`: the crossing times as the scope saw them (grey), and with the reference pickup subtracted (`remove_pickup`, colour). Compares noise, rise, overshoot and the settling times.

In [37]:
# Raw captures of the run (<csv stem>_waveforms/*.npz, pll_lock_time.py), processed twice: the crossing times as the
# scope saw them, and with remove_pickup (the reference pickup in the pad subtracted)
import ast
import warnings
from fractions import Fraction

# The analysis functions of pll_lock_time.py, taken from its source (the module itself imports the instrument drivers,
# which the notebook environment does not need): the same code that produced the CSV
SCRIPT = Path.cwd().resolve().parents[2] / 'sw' / 'tests' / 'pll_lock_time.py'
WANTED = {'TAIL_FRAC', 'STEP_TOL', 'PAD_F_TOL', 'SETTLE_BANDS', 'pickup_periods', 'crossing_times', 'remove_pickup',
          'frequency_curve', 'settled_value', 'settle_time', 'first_crossing', 'step_shape', 'capture_t50',
          'band_tol', 'scale_waveform', 'load_waveforms', 'load_reference', 'summarize'}
src = SCRIPT.read_text()
body = [n for n in ast.parse(src).body
        if (isinstance(n, ast.FunctionDef) and n.name in WANTED)
        or (isinstance(n, ast.Assign) and any(getattr(t, 'id', None) in WANTED for t in n.targets))]
plt_ns = dict(np=np, warnings=warnings, Fraction=Fraction, FB_DIV=128)  # FB_DIV: fixed feedback ratio (pll_spectrum)
exec(compile(ast.Module(body=body, type_ignores=[]), str(SCRIPT), 'exec'), plt_ns)
plt_mod = type('pll_lock_time', (), plt_ns)

WAVE_DIR = RUN_CSV.with_name(RUN_CSV.stem + '_waveforms')
wave_files = sorted(WAVE_DIR.glob('*.npz')) if WAVE_DIR.is_dir() else []
SHOW_NS = (-50, 700)  # [ns] around the step shown
SHOW_BANDS = {'10pct': '10 %', '5pct': '5 %', '2pct': '2 %', '1pct': '1 %'}  # of the step
VARIANTS = [('without pickup correction', False, 'rgba(120,120,120,0.9)'), ('with pickup correction', True, None)]
if not wave_files:
    print(f'no raw waveforms in {WAVE_DIR} (runs from before the raw capture saving have none)')

summary = []
for path in wave_files:
    meta, captures = plt_mod.load_waveforms(path)
    edges, total_div, periods = meta['edges'], int(meta['total_div']), int(meta['pickup_periods'])
    pos = str(meta['slope']) == 'POS'
    f_start = float(meta['f_vco']) if pos else plt_mod.FB_DIV * float(meta['f_hop'])
    step_exp = plt_mod.FB_DIV * (float(meta['f_hop']) - float(meta['f_ref'])) * (1 if pos else -1)
    crossings = [tc for tc in (plt_mod.crossing_times(t, v) for t, v in captures) if len(tc) >= 3]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.4, 0.6],
                        subplot_titles=('averaged response', 'frequency error |f - f_settled| in % of the step (log; dotted 1 %, dashed 2 %)'))
    for name, corrected, color in VARIANTS:
        curves = [plt_mod.frequency_curve(plt_mod.remove_pickup(tc, periods) if corrected else tc, edges, total_div)
                  for tc in crossings]
        fields, (mean, std, n) = plt_mod.summarize(curves, edges, step_exp, f_start)
        centres = (edges[:-1] + edges[1:]) / 2 * 1e9
        color = color or COLORS.get('up' if fields['direction'] == 'up' else 'down')
        fig.add_trace(go.Scatter(x=centres, y=mean / 1e9, mode='lines', name=name, legendgroup=name,
                                 line=dict(color=color, width=2)), row=1, col=1)
        err = (mean - fields['f_after']) / abs(fields['step_meas']) * 100  # [% of the step], 0 = settled value
        fig.add_trace(go.Scatter(x=centres, y=np.abs(err), mode='lines', name=name, legendgroup=name, showlegend=False,
                                 line=dict(color=color, width=2)), row=2, col=1)
        summary.append({'file': path.stem, 'processing': name, 'captures': fields['n_valid'],
                        'noise [%]': round(fields['noise_ppm'] / 1e4, 4),
                        'rise 10-90 [ns]': round(fields['t_rise_10_90'] * 1e9, 1),
                        'overshoot [%]': round(fields['overshoot_pct'], 1),
                        **{f't ±{v} [ns]': round(fields['t_settle_' + k] * 1e9, 1) for k, v in SHOW_BANDS.items()}})
    for band in (1, 2):
        fig.add_hline(y=band, line=dict(color='black', width=1, dash='dot' if band == 1 else 'dash'), opacity=0.6,
                      row=2, col=1)
    fig.update_xaxes(title='time after the step [ns]', range=SHOW_NS, row=2, col=1)
    fig.update_xaxes(range=SHOW_NS, row=1, col=1)
    fig.update_yaxes(title='VCO frequency [GHz]', row=1, col=1)
    fig.update_yaxes(title='|error| [% of step]', type='log', range=[-2.5, 2], row=2, col=1)
    fig.update_layout(title=f'{path.stem}: averaged response with and without the pickup correction '
                            f'({len(crossings)} captures)', height=800, template='plotly_white')
    save(fig, f'pickup_{path.stem}')
if summary:
    display(pd.DataFrame(summary))

saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_pickup_1of1_NEG.html and .png


saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_pickup_1of1_POS.html and .png


,file,processing,captures,noise [%],rise 10-90 [ns],overshoot [%],t ±10 % [ns],t ±5 % [ns],t ±2 % [ns],t ±1 % [ns]
0,1of1_NEG,without pickup correction,1000,0.1653,18.1,50.2,180.0,NaN,NaN,NaN
1,1of1_NEG,with pickup correction,1000,0.0042,19.0,44.6,140.0,180.0,1140.0,1980.0
2,1of1_POS,without pickup correction,1000,0.1028,29.8,39.5,220.0,NaN,NaN,NaN
3,1of1_POS,with pickup correction,1000,0.0072,30.2,39.2,220.0,220.0,290.0,550.0


## Reference pickup in the pad periods
The period error of the pad clock (each period minus the mean period, from the rising edges of the raw captures) before the step and in the settled end of the window, with the reference clock over it (the first captures of a run record `REF_CH`; older files have none, then only the period error is drawn). The 37.5 MHz reference couples into the pad cable. Its pattern repeats every 8 pad periods (the numerator of FB_DIV / total_div), is fixed in time and differs before and after the step (other reference frequency); the PLL jitter on top of it is random. It starts at a different edge in every capture, so the folded templates (bottom) are aligned before they are averaged. Needs the cell above (`plt_mod`, `wave_files`).

In [38]:
# The pickup in the time domain: the period error of the pad clock, i.e. (t_i - t_(i-1)) - mean period, per rising edge i,
# and the reference clock (REF_CH of pll_lock_time.py, recorded in the first captures) over it when the file has it.
# The mean period is that of a straight-line fit through the edges of the stretch (before the step; the last 20 % of
# the window, both settled), so the period error is the difference of the edge shifts from that line.
# Uses plt_mod and wave_files of the cell above.
N_USE = 300  # captures drawn / averaged per file
PRE_END = -0.1e-6  # [s] the edges before this are the pre-step stretch (as in remove_pickup)
plt_mod_ref_ch = 3  # REF_CH of the script (label only)
STRETCH_COLORS = {'before the step': '#2a6fdb', 'after the step (settled)': '#d1495b'}


def edge_shifts(tc, tail_frac=0.2):
    """{stretch: (edge index, edge time [s], deviation from the line through the stretch [s])} of one capture."""
    idx = np.arange(len(tc))
    sel = {'before the step': tc < PRE_END, 'after the step (settled)': idx >= len(tc) - int(round(tail_frac * len(tc)))}
    out = {}
    for name, m in sel.items():
        slope, icpt = np.polyfit(idx[m], tc[m], 1)
        out[name] = (idx[m], tc[m], tc[m] - (icpt + slope * idx[m]))
    return out


def period_errors(shifts):
    """(edge index, edge time [s], period error [s]) of one stretch of edge_shifts: the period ending at each edge."""
    idx, tc, dev = shifts
    return idx[1:], tc[1:], np.diff(dev)


def fold(shifts, P):
    """Mean period error per position in the P-edge pattern (mean removed)."""
    idx, _, dev = period_errors(shifts)
    tpl = np.array([dev[idx % P == k].mean() for k in range(P)])
    return tpl - tpl.mean()


def align(templates):
    """The templates circularly shifted onto each other (the pattern starts at a different edge in every capture):
    first onto the first one, then onto the mean of those."""
    ref = templates[0]
    for _ in range(2):
        aligned = np.array([np.roll(t, max(range(len(t)), key=lambda k: np.dot(np.roll(t, k), ref)))
                            for t in templates])
        ref = aligned.mean(axis=0)
    return aligned


for path in wave_files:
    meta, captures = plt_mod.load_waveforms(path)
    P = int(meta['pickup_periods'])
    crossings = [tc for tc in (plt_mod.crossing_times(t, v) for t, v in captures[:N_USE]) if len(tc) >= 3]
    shifts = [edge_shifts(tc) for tc in crossings]
    ref = plt_mod.load_reference(path)
    k_ref = min(ref) if ref else None  # the capture shown: the first with a reference record
    j_show = list(meta['capture']).index(k_ref) if ref and k_ref in list(meta['capture']) else 0
    if not ref:
        print(f'{path.stem}: no reference records in the file (REF_CH of the script was off): period error only')
    fig = make_subplots(rows=2, cols=2, vertical_spacing=0.16, row_heights=[0.45, 0.55],
                        specs=[[{'secondary_y': True}, {'secondary_y': True}], [{'colspan': 2}, None]],
                        subplot_titles=('one capture: period error before the step', 'same capture: after the step (settled)',
                                        f'folded onto the {P}-edge pattern (one marker per capture, line: mean of '
                                        f'{len(crossings)} aligned captures)'))
    notes = []
    for col, (name, color) in enumerate(STRETCH_COLORS.items(), start=1):
        idx, tc, dev = period_errors(shifts[j_show][name])
        if ref:  # the reference over the same stretch (right axis, in V)
            t_r, v_r = ref[k_ref]
            m = (t_r >= tc[0] - 5e-9) & (t_r <= tc[-1] + 5e-9)
            fig.add_trace(go.Scatter(x=t_r[m] * 1e9, y=v_r[m], mode='lines', name='reference (CH%d, right axis)' % plt_mod_ref_ch,
                                     legendgroup='ref', showlegend=(col == 1), line=dict(color='#555555', width=1)),
                          row=1, col=col, secondary_y=True)
        fig.add_trace(go.Scatter(x=tc * 1e9, y=dev * 1e12, mode='lines+markers', name=name, legendgroup=name,
                                 line=dict(color=color, width=1.5), marker=dict(size=5)), row=1, col=col)
        aligned = align(np.array([fold(s[name], P) for s in shifts])) * 1e12
        pos = np.arange(P)
        for a in aligned:
            fig.add_trace(go.Scatter(x=pos, y=a, mode='markers', legendgroup=name, showlegend=False,
                                     marker=dict(size=4, color=color, opacity=0.12)), row=2, col=1)
        mean = aligned.mean(axis=0)
        fig.add_trace(go.Scatter(x=pos, y=mean, mode='lines+markers', legendgroup=name, showlegend=False,
                                 line=dict(color=color, width=3), marker=dict(size=9, color=color)), row=2, col=1)
        notes.append(f'{name}: pattern {np.ptp(mean):.0f} ps peak to peak, spread between captures '
                     f'{aligned.std(axis=0).mean():.0f} ps, single period {np.std(dev) * 1e12:.0f} ps rms in this capture')
    for col in (1, 2):
        fig.update_xaxes(title='time after the step [ns]', row=1, col=col)
    fig.update_xaxes(title='edge position in the pattern', dtick=1, row=2, col=1)
    fig.update_yaxes(title='period error [ps]', row=1, col=1, secondary_y=False)
    fig.update_yaxes(title='period error [ps]', row=2, col=1)
    if ref:
        fig.update_yaxes(title='reference [V]', row=1, col=2, secondary_y=True)
    fig.update_layout(title=f'{path.stem}: reference pickup in the periods of the pad clock', height=800, template='plotly_white')
    print('\n'.join(f'{path.stem}, {n}' for n in notes))
    save(fig, f'pickup_edges_{path.stem}')

1of1_NEG, before the step: pattern 1506 ps peak to peak, spread between captures 8 ps, single period 444 ps rms in this capture
1of1_NEG, after the step (settled): pattern 1526 ps peak to peak, spread between captures 20 ps, single period 543 ps rms in this capture
saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_pickup_edges_1of1_NEG.html and .png


1of1_POS, before the step: pattern 1528 ps peak to peak, spread between captures 9 ps, single period 547 ps rms in this capture
1of1_POS, after the step (settled): pattern 1502 ps peak to peak, spread between captures 5 ps, single period 458 ps rms in this capture
saved /home/willem/projects/lagd-meas/results/pll/pll_lock_time_S5_20261109_230454_pickup_edges_1of1_POS.html and .png
